In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()


路径

In [20]:
data_path = "dataset\processed\Brazilian-flights.csv"

df = pd.read_csv(data_path, encoding="latin1")
print(f"数据规模: {df.shape[0]} 行 * {df.shape[1]} 列")


<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\13310\AppData\Local\Temp\ipykernel_12980\4111500782.py:1: SyntaxWarning: invalid escape sequence '\p'
  data_path = "dataset\processed\Brazilian-flights.csv"


数据规模: 2542519 行 * 20 列


In [21]:
new_columns = {
    "Flight.No": "航班号",
    "Airline": "航空公司",
    "Flight.Type": "航班类型",
    "Scheduled.Departure": "计划起飞时间",
    "Departure": "实际起飞时间",
    "Scheduled.Arrival": "计划到达时间",
    "Arrival": "实际到达时间",
    "Flight.Status": "航班状态",
    "Justification": "延误说明",
    "Airport.From": "出发机场",
    "Airport.To": "到达机场",
    "Longitude.To": "到达经度",
    "Latitude.To": "到达纬度",
    "Longitude.From": "出发经度",
    "Latitude.From": "出发纬度",
    "Departure.Delay": "起飞延误",
    "Arrival.Delay": "到达延误",
    "Distance.In.Meters": "航程距离",
    "Flight.No2": "航班编号",
    "AirLine.Code": "航空公司代码",
}

df = df.rename(columns=new_columns)
display(df.head())


,航班号,航空公司,航班类型,计划起飞时间,实际起飞时间,计划到达时间,实际到达时间,航班状态,延误说明,出发机场,到达机场,到达经度,到达纬度,出发经度,出发纬度,起飞延误,到达延误,航程距离,航班编号,航空公司代码
0,AAL - 203,AMERICAN AIRLINES INC,International,2016-01-30 08:58:00,2016-01-30 08:58:00,2016-01-30 10:35:00,2016-01-30 10:35:00,Confirmed,NaN,SBCT,SBPA,-51.175381,-29.993473,-49.172481,-25.532713,0.0,0.0,5.322599e+05,203,AAL
1,AAL - 203,AMERICAN AIRLINES INC,International,2016-01-13 12:13:00,2016-01-13 12:13:00,2016-01-13 21:30:00,2016-01-13 21:30:00,Confirmed,NaN,SBPA,KMIA,-80.287046,25.795865,-51.175381,-29.993473,0.0,0.0,6.910629e+06,203,AAL
2,AAL - 203,AMERICAN AIRLINES INC,International,2016-01-29 12:13:00,2016-01-29 12:13:00,2016-01-29 21:30:00,2016-01-29 21:30:00,Confirmed,NaN,SBPA,KMIA,-80.287046,25.795865,-51.175381,-29.993473,0.0,0.0,6.910629e+06,203,AAL
3,AAL - 203,AMERICAN AIRLINES INC,International,2016-01-19 12:13:00,2016-01-18 12:03:00,2016-01-19 21:30:00,2016-01-18 20:41:00,Confirmed,AIR TRAFFIC LIBERATION/ANTECIPATION,SBPA,KMIA,-80.287046,25.795865,-51.175381,-29.993473,-1450.0,-1489.0,6.910629e+06,203,AAL
4,AAL - 203,AMERICAN AIRLINES INC,International,2016-01-30 12:13:00,2016-01-30 12:13:00,2016-01-30 21:30:00,2016-01-30 21:30:00,Confirmed,NaN,SBPA,KMIA,-80.287046,25.795865,-51.175381,-29.993473,0.0,0.0,6.910629e+06,203,AAL


In [22]:
missing = df.isna().sum().sort_values(ascending=False)
print("缺失值统计：")
display(missing[missing > 0])

print("重复行数：", df.duplicated().sum())


缺失值统计：


延误说明      1510212
起飞延误       289196
实际起飞时间     289196
实际到达时间     289196
到达延误       289196
dtype: int64

重复行数： 498


In [23]:
#去除重复行
df = df.drop_duplicates().reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2542021 entries, 0 to 2542020
Data columns (total 20 columns):
 #   Column  Dtype  
---  ------  -----  
 0   航班号     object 
 1   航空公司    object 
 2   航班类型    object 
 3   计划起飞时间  object 
 4   实际起飞时间  object 
 5   计划到达时间  object 
 6   实际到达时间  object 
 7   航班状态    object 
 8   延误说明    object 
 9   出发机场    object 
 10  到达机场    object 
 11  到达经度    float64
 12  到达纬度    float64
 13  出发经度    float64
 14  出发纬度    float64
 15  起飞延误    float64
 16  到达延误    float64
 17  航程距离    float64
 18  航班编号    int64  
 19  航空公司代码  object 
dtypes: float64(7), int64(1), object(12)
memory usage: 387.9+ MB


In [25]:
# 统计起飞延误列的负数数量
departure_neg_count = (df['起飞延误'] < 0).sum()
print(f"起飞延误负数数量: {departure_neg_count}")

# 统计到达延误列的负数数量
arrival_neg_count = (df['到达延误'] < 0).sum()
print(f"到达延误负数数量: {arrival_neg_count}")

# 两列都是负数的行数
both_neg_count = ((df['起飞延误'] < 0) & (df['到达延误'] < 0)).sum()
print(f"两列都是负数的行数: {both_neg_count}")

# 负延误原因
neg_delay = df[(df["起飞延误"] < 0) | (df["到达延误"] < 0)]
reason_counts = (
    neg_delay["延误说明"]
    .fillna("未知原因")
    .value_counts(dropna=False)
)

print("负延误对应的原因：")
print(reason_counts)


起飞延误负数数量: 376162
到达延误负数数量: 411783
两列都是负数的行数: 347854
负延误对应的原因：
延误说明
ANTECIPATION                                              352774
AIR TRAFFIC LIBERATION/ANTECIPATION                        34655
未知原因                                                       18985
DELAY NOT SPECIFIED - OTHER                                11394
AIRPORT UNDER OPERATIONAL RESTRICTIONS                      7451
ARRIVAL AIRPORT NOT OPERATING                               2399
AIRPLANE CONECTION                                          2090
ANTECIPATION - INTERNATIONAL                                1410
SECURITY/PASSENGER/CARGO/ALARM                              1032
AIRPLANE FAULT                                               978
ARRIVAL AIRPORT UNDER LIMITS                                 952
DEPARTURE AIRPORT NOT OPERATING                              903
CONECTION - FLIGHT CANCELLED - AIPORT NOT OPERATING          752
CANCELLED - CONECTION - FLIGHT CANCELLED - BAD WEATHER       726
AIRCRAFT EXCHANGE      